# Genotype, copy number, and generated cells in the AML mosaic latent space

This experimental tutorial asks what the AML mosaic latent space can and cannot say about somatic genotype.
It reuses the unrefined RNA–protein bridge from the [Fig. 7 notebook](../../reproducibility/api/fig7_aml.ipynb)
when available. A fresh runtime can instead train that base bridge here, using the same public API,
sample split, preprocessing, and model settings. Fig. 7 mutation refinement and cross-validation are not needed.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/tutorials/experimental/aml_genotype_latent.ipynb)

**Choose a bridge source before Run all.** An existing cache is reused first. To reuse the exact Fig. 7 model
in a different Colab session, upload its `univi_aml_bridge_reference.zip` with the Files panel (or set
`BRIDGE_ARCHIVE_PATH` for a custom path). If no reference is available, `TRAIN_BRIDGE_IF_MISSING=True`
trains and caches the base bridge. This is real GPU training and uses runtime/compute units; set it to False
if you only want to load a reference. No pretrained public download is assumed.

Local notebooks share the cache even when started in different directories. Colab disks and their caches
are temporary and are not shared between runtimes. For persistent reuse, mount Drive and set
`UNIVI_AML_REFERENCE_DIR` to a Drive folder before running the settings cell in both notebooks.

**Resources.** The full all-gene analysis and CNV step can require a high-memory runtime; GPU memory and system
RAM are separate. Reusing a reference skips training, but query preprocessing and CNV still need RAM.

Sections

1. What genotype information the frozen bridge carries, conditioned on patient and cell state
2. Joint clone genotypes in DAb-seq and where they sit in the latent space
3. Expression-inferred copy-number subclones in van Galen, overlaid on mutation calls and the latent space
4. Sampling genotype-defined regions of the latent space and decoding them, with realism checks
5. A cross-modal counterfactual: a genotype direction learned from protein+genotype, read out as RNA
6. What these analyses support, and what they do not

Throughout, "genotype signal" means signal that survives conditioning on sample and cell state.

Extra dependency for section 3: `pip install "infercnvpy[gtf]>=0.6,<0.7"`; the GENCODE annotation is downloaded once
into the UniVI cache.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "univi[tutorials]==1.2.1" "infercnvpy[gtf]>=0.6,<0.7" "pandas==2.2.3"
    import importlib.metadata as _md
    if "pandas" in sys.modules and sys.modules["pandas"].__version__ != _md.version("pandas"):
        print("pandas changed after it was imported: use Runtime > Restart session, then run all cells again.")

In [ ]:
import warnings

# Cosmetic warnings from optional notebook widgets and deprecations in dependencies.
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning, module="scanpy")
warnings.filterwarnings("ignore", category=FutureWarning, module="anndata")
warnings.filterwarnings("ignore", category=FutureWarning, module="infercnvpy")

import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact, spearmanr

import univi.datasets as uds
from univi.evaluation import encode_adata, fit_label_latent_gaussians, generate_from_latent, sample_latent_by_label
from univi.utils.seed import set_seed
from univi.workflows import load_reference

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
set_seed(0)
import os
BRIDGE_DIR = Path(os.environ.get(
    "UNIVI_AML_REFERENCE_DIR",
    str(uds.get_data_dir() / "references" / "univi_aml_bridge_reference"),
)).expanduser()
BRIDGE_ARCHIVE_PATH = None   # optional path to a ZIP exported by Fig. 7
TRAIN_BRIDGE_IF_MISSING = True
BRIDGE_EPOCHS = 3000         # Fig. 7 limit; early stopping is enabled
BRIDGE_BATCH_SIZE = 256

HERO = "NPM1"
GENES = ["NPM1", "DNMT3A", "FLT3", "TP53", "NRAS", "TET2", "IDH2"]
DAB_GENES = ["NPM1", "DNMT3A", "FLT3"]

# van Galen samples are integer codes in `orig.ident`; optionally map them to patients (see the Fig. 7 notebook).
VG_SAMPLE_TO_PATIENT = None
# Out-of-fold mutation-head predictions written by the Fig. 7 cross-validation (optional; used in section 1).
OOF_PATH = BRIDGE_DIR.parent / "aml_fig7_cv_oof_predictions.csv.gz"
print("device:", device)

### Reuse or build the base bridge

The next cells define reference discovery and the fallback training recipe. Training happens only in the later
load cell if no complete reference or supplied ZIP is available. The ZIP must include the fitted preprocessors.


In [ ]:
# Reference discovery and ZIP import use the standard library only.
import json
import shutil
import tempfile
import zipfile
from pathlib import PurePosixPath

REFERENCE_FILES = ("model.pt", "preprocessing.joblib", "metadata.json")

def complete_reference(directory):
    directory = Path(directory)
    return all((directory / name).is_file() and (directory / name).stat().st_size > 0
               for name in REFERENCE_FILES)

def import_bridge_zip(archive_path, destination):
    """Import the three reference files without extracting arbitrary archive paths."""
    destination = Path(destination)
    if destination.exists():
        raise FileExistsError(f"Choose a new BRIDGE_DIR; refusing to overwrite {destination}.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        selected = {}
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or ".." in path.parts or "\\" in info.filename:
                raise ValueError("The reference ZIP contains an unsafe path.")
            if not info.is_dir() and path.name in REFERENCE_FILES:
                if path.name in selected:
                    raise ValueError(f"The ZIP has multiple copies of {path.name}.")
                selected[path.name] = info
        if set(selected) != set(REFERENCE_FILES):
            raise ValueError(f"The ZIP must contain {REFERENCE_FILES}, including fitted preprocessors.")
        if len({PurePosixPath(info.filename).parent for info in selected.values()}) != 1:
            raise ValueError("The three reference files must be from one folder.")
        with tempfile.TemporaryDirectory(prefix=".univi_reference_tmp_", dir=destination.parent) as tmp:
            staged = Path(tmp) / "reference"
            staged.mkdir()
            for name, info in selected.items():
                with archive.open(info) as src, (staged / name).open("wb") as dst:
                    shutil.copyfileobj(src, dst)
            metadata = json.loads((staged / "metadata.json").read_text())
            if metadata.get("dataset") != "aml_mosaic":
                raise ValueError("This ZIP is not an aml_mosaic reference.")
            if not complete_reference(staged):
                raise ValueError("The reference ZIP contains empty files.")
            staged.rename(destination)
    return destination

def resolve_aml_reference(destination, *, archive_path=None, train_if_missing=True, builder=None):
    destination = Path(destination).expanduser()
    if complete_reference(destination):
        print("Reusing cached bridge:", destination.name)
        return destination
    if destination.exists():
        raise RuntimeError(f"Incomplete reference at {destination}. Choose another BRIDGE_DIR or repair it.")
    if archive_path is not None:
        return import_bridge_zip(Path(archive_path).expanduser(), destination)

    # Recognize Fig. 7's older working-directory-relative output as well.
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / "univi_aml_bridge_reference",
                          root / "docs/reproducibility/api/univi_aml_bridge_reference"):
            if complete_reference(candidate):
                print("Reusing legacy Fig. 7 bridge:", candidate.name)
                return candidate

    candidates = [Path.cwd() / "univi_aml_bridge_reference.zip",
                  Path.cwd() / "univi_outputs/aml/univi_aml_bridge_reference.zip"]
    if "google.colab" in sys.modules:
        candidates.append(Path("/content/univi_aml_bridge_reference.zip"))
    for candidate in dict.fromkeys(p.resolve() for p in candidates):
        if candidate.is_file():
            print("Importing bridge ZIP:", candidate.name)
            return import_bridge_zip(candidate, destination)
    if not train_if_missing:
        raise FileNotFoundError(
            "No AML bridge found. Upload the ZIP exported by Fig. 7 and set BRIDGE_ARCHIVE_PATH, "
            "point BRIDGE_DIR to its complete reference folder, or enable TRAIN_BRIDGE_IF_MISSING."
        )
    if builder is None:
        raise ValueError("Missing bridge builder.")
    print("No saved bridge found: training the base CITE-seq bridge once. This uses runtime compute.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".univi_reference_tmp_", dir=destination.parent) as tmp:
        staged = Path(tmp) / "reference"
        builder(staged)
        if not complete_reference(staged):
            raise RuntimeError("Training did not produce a complete reference.")
        staged.rename(destination)
    return destination


In [ ]:
def build_base_aml_bridge(directory):
    """Train only Fig. 7's unrefined bridge; no mutation labels or CV are used."""
    from univi import ModalityConfig, TrainingConfig, UniVIConfig, UniVIMultiModalVAE, UniVITrainer
    from univi.preprocessing import ADTPreprocessor, RNAPreprocessor
    from univi.workflows import make_loader, save_reference
    from importlib.metadata import version

    set_seed(0)
    paired = uds.load("aml_mosaic", keys=["cite_rna", "cite_adt"])
    rna, adt = paired["cite_rna"], paired["cite_adt"]
    if not rna.obs_names.equals(adt.obs_names):
        raise ValueError("CITE RNA and protein cell order differs.")
    if "split" in rna.obs:
        splits = {k: np.flatnonzero(rna.obs["split"].to_numpy() == k) for k in ("train", "val", "test")}
    else:
        samples = np.random.default_rng(0).permutation(rna.obs["sample_id"].astype(str).unique())
        group = rna.obs["sample_id"].astype(str).to_numpy()
        splits = {"test": np.flatnonzero(np.isin(group, samples[:2])),
                  "val": np.flatnonzero(np.isin(group, samples[2:3])),
                  "train": np.flatnonzero(np.isin(group, samples[3:]))}
    if not len(splits["train"]) or not len(splits["val"]):
        raise ValueError("The bridge needs nonempty training and validation splits.")
    rp = RNAPreprocessor(n_hvg=None, scale=True).fit(rna[splits["train"]])
    ap = ADTPreprocessor(scale=True, clip=10.0).fit(adt[splits["train"]])
    # Transform only the splits needed for base training, avoiding a dense test copy.
    train = {"rna": rp.transform(rna[splits["train"]]), "adt": ap.transform(adt[splits["train"]])}
    val = {"rna": rp.transform(rna[splits["val"]]), "adt": ap.transform(adt[splits["val"]])}
    cfg = UniVIConfig(
        latent_dim=30, beta=1.15, gamma=1.75, encoder_dropout=0.10, decoder_dropout=0.05,
        modalities=[
            ModalityConfig("rna", train["rna"].n_vars, [1024, 512, 256, 128, 64],
                           [64, 128, 256, 512, 1024], likelihood="gaussian"),
            ModalityConfig("adt", train["adt"].n_vars, [128, 64, 32], [32, 64, 128], likelihood="gaussian"),
        ],
    )
    base = UniVIMultiModalVAE(cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
    trainer = UniVITrainer(
        base,
        make_loader(train, batch_size=BRIDGE_BATCH_SIZE, shuffle=True, drop_last=True),
        make_loader(val, batch_size=1024),
        TrainingConfig(n_epochs=BRIDGE_EPOCHS, batch_size=BRIDGE_BATCH_SIZE,
                       lr=1e-4, weight_decay=1e-5, device=device,
                       early_stopping=True, patience=150, log_every=100),
    )
    result = trainer.fit()
    save_reference(directory, base, preprocessors={"rna": rp, "adt": ap}, metadata={
        "dataset": "aml_mosaic", "notebook": "aml_genotype_latent", "recipe": "fig7_base_bridge",
        "stage": "base_before_mutation_refinement", "seed": 0,
        "split": "cite_rna.obs['split'] or seed-0 sample split",
        "n_epochs_limit": BRIDGE_EPOCHS, "batch_size": BRIDGE_BATCH_SIZE,
        "versions": {name: version(name) for name in ("univi", "torch", "numpy", "scikit-learn")},
    })
    print("Saved base bridge:", directory)


In [ ]:
# Helper functions for this notebook (numpy / pandas / scikit-learn / matplotlib only).
from typing import Mapping, Sequence
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 200})



def stratified_auroc(y, p, strata, min_pos=5, min_neg=5):
    """AUROC computed inside each stratum (e.g. patient x cell state), then averaged.

    A pooled AUROC can be high purely because genotype prevalence differs between
    patients or cell states. The within-stratum average asks whether the score ranks
    mutant above wild-type cells *within* comparable cells.
    Returns (summary dict, per-stratum DataFrame).
    """
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    strata = np.asarray(strata).astype(str)
    ok = np.isfinite(y) & np.isfinite(p)
    rows = []
    for s in np.unique(strata[ok]):
        m = ok & (strata == s)
        n_pos, n_neg = int((y[m] == 1).sum()), int((y[m] == 0).sum())
        if n_pos >= min_pos and n_neg >= min_neg:
            rows.append({"stratum": s, "n_pos": n_pos, "n_neg": n_neg,
                         "auroc": roc_auc_score(y[m], p[m])})
    per = pd.DataFrame(rows, columns=["stratum", "n_pos", "n_neg", "auroc"])
    if per.empty:
        return {"n_strata": 0, "weighted_auroc": np.nan, "unweighted_auroc": np.nan}, per
    w = 1.0 / (1.0 / per["n_pos"] + 1.0 / per["n_neg"])  # harmonic-mean-like weight
    summary = {
        "n_strata": int(len(per)),
        "cells_used": int((per["n_pos"] + per["n_neg"]).sum()),
        "cells_labeled": int(ok.sum()),
        "weighted_auroc": float(np.average(per["auroc"], weights=w)),
        "unweighted_auroc": float(per["auroc"].mean()),
    }
    return summary, per


def group_bootstrap(y, p, groups, metric=roc_auc_score, n_boot=1000, seed=0):
    """Cluster (group) bootstrap CI: resample whole patients/experiments with replacement."""
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    g = np.asarray(groups).astype(str)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p, g = y[ok], p[ok], g[ok]
    uniq = np.unique(g)
    point = float(metric(y, p)) if len(np.unique(y)) == 2 else np.nan
    if len(uniq) < 2:
        return {"point": point, "lo": np.nan, "hi": np.nan, "n_groups": int(len(uniq)), "n_valid_boot": 0}
    rng = np.random.default_rng(seed)
    members = {u: np.flatnonzero(g == u) for u in uniq}
    stats = []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([members[u] for u in pick])
        if len(np.unique(y[idx])) == 2:
            stats.append(metric(y[idx], p[idx]))
    lo, hi = (np.percentile(stats, [2.5, 97.5]) if stats else (np.nan, np.nan))
    return {"point": point, "lo": float(lo), "hi": float(hi),
            "n_groups": int(len(uniq)), "n_valid_boot": int(len(stats))}


def covariate_only_baseline(y, covariate, groups, prior_strength=1.0):
    """Leave-one-group-out target encoding: P(mut | covariate) learned on other groups.

    Use a cell-state/cluster label as the covariate. If this baseline matches the
    mutation head, the head is mostly reading cell state, not genotype.
    """
    y = np.asarray(y, dtype=float)
    c = np.asarray(covariate).astype(str)
    g = np.asarray(groups).astype(str)
    out = np.full(len(y), np.nan)
    lab = np.isfinite(y)
    for grp in np.unique(g[lab]):
        test = lab & (g == grp)
        train = lab & (g != grp)
        if not train.any():
            continue
        prior = y[train].mean()
        df = pd.DataFrame({"c": c[train], "y": y[train]}).groupby("c")["y"].agg(["sum", "count"])
        rate = (df["sum"] + prior_strength * prior) / (df["count"] + prior_strength)
        out[test] = pd.Series(c[test]).map(rate).fillna(prior).to_numpy()
    return out


def grouped_linear_probe(z, y, groups, n_splits=5, C=1.0, seed=0):
    """Out-of-fold logistic-regression probabilities with whole groups held out.

    A frozen-latent linear probe is the baseline a mutation head must beat.
    """
    z = np.asarray(z); y = np.asarray(y, dtype=float); g = np.asarray(groups).astype(str)
    lab = np.isfinite(y)
    out = np.full(len(y), np.nan)
    n_groups = len(np.unique(g[lab]))
    if n_groups < 2:
        raise ValueError("Need at least two labeled groups for a grouped probe.")
    cv = GroupKFold(n_splits=min(n_splits, n_groups))
    zl, yl, gl = z[lab], y[lab], g[lab]
    idx_lab = np.flatnonzero(lab)
    for tr, te in cv.split(zl, yl, gl):
        if len(np.unique(yl[tr])) < 2:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=seed))
        clf.fit(zl[tr], yl[tr])
        out[idx_lab[te]] = clf.predict_proba(zl[te])[:, 1]
    return out


def neighborhood_genotype_enrichment(z, y01, groups, k=30, n_perm=200, seed=0) -> dict:
    """kNN genotype agreement vs. a global null and a within-group (patient) null.

    agreement = mean over labeled cells of the fraction of their k labeled neighbours
    that share their genotype. Shuffling labels globally destroys all structure;
    shuffling *within* each group keeps each patient's mutant prevalence, so
      observed - within_group_null  = genotype structure beyond patient identity
      within_group_null - global_null = structure explained by patient composition.
    """
    y = np.asarray(y01, dtype=float)
    lab = np.isfinite(y)
    zl, yl = np.asarray(z)[lab], y[lab]
    gl = np.asarray(groups).astype(str)[lab]
    idx = NearestNeighbors(n_neighbors=min(k + 1, len(yl))).fit(zl).kneighbors(zl, return_distance=False)[:, 1:]

    def agreement(labels):
        return float((labels[idx] == labels[:, None]).mean())

    rng = np.random.default_rng(seed)
    obs = agreement(yl)
    glob = np.array([agreement(rng.permutation(yl)) for _ in range(n_perm)])
    members = [np.flatnonzero(gl == u) for u in np.unique(gl)]
    within = []
    for _ in range(n_perm):
        yp = yl.copy()
        for m in members:
            yp[m] = rng.permutation(yl[m])
        within.append(agreement(yp))
    within = np.array(within)
    return {
        "n_cells": int(len(yl)), "k": int(idx.shape[1]),
        "observed": obs,
        "global_null_mean": float(glob.mean()),
        "within_group_null_mean": float(within.mean()),
        "excess_over_within_group_null": float(obs - within.mean()),
        "z_vs_within_group_null": float((obs - within.mean()) / (within.std(ddof=1) + 1e-12)),
        "p_vs_within_group_null": float((1 + (within >= obs).sum()) / (1 + n_perm)),
    }


LSC17_WEIGHTS = {
    "DNMT3B": 0.0874, "ZBTB46": -0.0347, "NYNRIN": 0.00865, "ARHGAP22": -0.0138,
    "LAPTM4B": 0.00582, "MMRN1": 0.0258, "DPYSL3": 0.0284, "KIAA0125": 0.0196,
    "CDK6": -0.0704, "CPXM1": -0.0258, "SOCS2": 0.0271, "SMIM24": -0.0226,
    "EMP1": 0.0146, "NGFRAP1": 0.0465, "CD34": 0.0338, "AKR1C3": -0.0402, "GPR56": 0.0501,
}


LSC17_ALIASES = {"KIAA0125": "FAM30A", "NGFRAP1": "BEX3", "GPR56": "ADGRG1"}


def lsc17_weighted(X, gene_names, *, standardize=True) -> dict:
    """Weighted LSC17 (published coefficients) on log-normalized single-cell expression.

    X: cells x genes (dense or scipy sparse), log-normalized. Genes are matched by the
    original or current symbol. With standardize=True each gene is z-scored across the
    supplied cells first (the bulk score was defined on scaled expression). Applying a
    bulk-derived prognostic score per cell is a heuristic; report it as such.
    """
    import scipy.sparse as sp
    names = pd.Index(np.asarray(gene_names).astype(str))
    cols, w, used = [], [], []
    for g, coef in LSC17_WEIGHTS.items():
        for cand in (g, LSC17_ALIASES.get(g)):
            if cand is not None and cand in names:
                cols.append(names.get_loc(cand)); w.append(coef); used.append(cand)
                break
    if not cols:
        raise ValueError("No LSC17 genes found in gene_names.")
    M = X[:, cols]
    M = M.toarray() if sp.issparse(M) else np.asarray(M, dtype=float)
    if standardize:
        M = (M - M.mean(0)) / (M.std(0) + 1e-8)
    return {"score": M @ np.asarray(w), "genes_used": used, "n_genes": len(used)}


def knn_distance_ratio(z_gen, z_real, k=15, seed=0) -> dict:
    """Median distance from generated points to real cells, relative to real-to-real.

    Splits the real cells in half: A is the reference, B gives the real-to-real baseline.
    Ratio ~1 means generated latents sit on the data manifold; >>1 means off-manifold.
    """
    rng = np.random.default_rng(seed)
    z_real = np.asarray(z_real)
    perm = rng.permutation(len(z_real))
    a, b = z_real[perm[: len(perm) // 2]], z_real[perm[len(perm) // 2:]]
    nn = NearestNeighbors(n_neighbors=k).fit(a)
    d_gen = nn.kneighbors(np.asarray(z_gen))[0].mean(1)
    d_real = nn.kneighbors(b)[0].mean(1)
    return {"median_gen": float(np.median(d_gen)), "median_real": float(np.median(d_real)),
            "ratio": float(np.median(d_gen) / np.median(d_real))}


def c2st_auc(x_a, x_b, n_splits=5, seed=0, max_n=5000) -> float:
    """Classifier two-sample test: cross-validated AUROC separating set A from set B.

    0.5 = indistinguishable. Compare generated-then-decoded cells with *decoded real
    cells* (same decoder) so the test isolates the latent sampling step.
    """
    rng = np.random.default_rng(seed)
    x_a, x_b = np.asarray(x_a), np.asarray(x_b)
    if len(x_a) > max_n: x_a = x_a[rng.choice(len(x_a), max_n, replace=False)]
    if len(x_b) > max_n: x_b = x_b[rng.choice(len(x_b), max_n, replace=False)]
    X = np.vstack([x_a, x_b]); y = np.r_[np.zeros(len(x_a)), np.ones(len(x_b))]
    aucs = []
    for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(X, y):
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        clf.fit(X[tr], y[tr]); aucs.append(roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1]))
    return float(np.mean(aucs))


def stratified_mean_difference_direction(z, y01, strata, min_per_class=10) -> dict:
    """Genotype direction in latent space estimated *within* cell-state strata.

    For each stratum with enough mutant and wild-type cells, take mean(z_mut) - mean(z_wt);
    average the differences with harmonic-mean weights. Conditioning on cell state keeps
    the direction from simply pointing from normal lineages to blasts.
    """
    z = np.asarray(z); y = np.asarray(y01, dtype=float); s = np.asarray(strata).astype(str)
    rows, diffs, weights = [], [], []
    for st in np.unique(s):
        m = (s == st) & np.isfinite(y)
        n1, n0 = int((y[m] == 1).sum()), int((y[m] == 0).sum())
        if n1 >= min_per_class and n0 >= min_per_class:
            d = z[m & (y == 1)].mean(0) - z[m & (y == 0)].mean(0)
            w = 1.0 / (1.0 / n1 + 1.0 / n0)
            diffs.append(d); weights.append(w)
            rows.append({"stratum": st, "n_mut": n1, "n_wt": n0, "norm": float(np.linalg.norm(d))})
    if not diffs:
        raise ValueError("No stratum has enough mutant and wild-type cells.")
    d = np.average(np.vstack(diffs), axis=0, weights=weights)
    cos = [float(np.dot(x, d) / (np.linalg.norm(x) * np.linalg.norm(d) + 1e-12)) for x in diffs]
    table = pd.DataFrame(rows); table["cosine_to_pooled"] = cos
    return {"direction": d / (np.linalg.norm(d) + 1e-12), "raw": d, "per_stratum": table}


GREY_LABELS = {"Unlabeled", "Other", "Excluded", "Unknown", "Unassigned", "General PBMC", "nan", "NA", "None"}


FIXED_COLORS = {"MUT": "#d62728", "WT": "#6baed6"}


GREY = "#d9d9d9"


def _is_grey(hex_color, tol=0.06):
    from matplotlib.colors import to_rgb
    r, g, b = to_rgb(hex_color)
    return max(r, g, b) - min(r, g, b) < tol


def _palette(categories):
    """Distinct colours for real categories; light grey for unlabeled/excluded; fixed MUT/WT colours."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_hex
    base = []
    for name in ("tab10", "Dark2", "Set2", "tab20b", "Set1"):
        base += [to_hex(c) for c in plt.get_cmap(name).colors]
    base = [c for c in dict.fromkeys(base) if not _is_grey(c)]
    cats = sorted(c for c in {str(x) for x in categories} if c not in GREY_LABELS and c not in FIXED_COLORS)
    pal = {c: base[i % len(base)] for i, c in enumerate(cats)}
    pal.update(FIXED_COLORS)
    pal.update({c: GREY for c in GREY_LABELS})
    return pal


def _point_size(n):
    return float(np.clip(60000.0 / max(n, 1), 0.3, 8.0))


def plot_embedding(ax, xy, values, *, title="", categorical=None, palette=None, s=None, cmap="viridis",
                   vmin=None, vmax=None, legend=True, top=("MUT",), seed=0, rasterized=True):
    """One embedding panel.

    Grey (unlabeled / excluded) cells are drawn first; all other cells are drawn in one random order so no
    category systematically hides another; categories listed in ``top`` (default MUT) are drawn last.
    """
    xy = np.asarray(xy)
    n = len(xy)
    s = _point_size(n) if s is None else s
    v = pd.Series(values).reset_index(drop=True)
    if categorical is None:
        categorical = not pd.api.types.is_numeric_dtype(v)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    rng = np.random.default_rng(seed)
    if categorical:
        v = v.astype(str).to_numpy()
        palette = palette or _palette(v)
        grey = np.isin(v, list(GREY_LABELS))
        on_top = np.isin(v, list(top)) & ~grey
        mid = ~grey & ~on_top
        for m in (grey, mid, on_top):
            idx = np.flatnonzero(m)
            idx = idx[rng.permutation(len(idx))]
            if len(idx):
                ax.scatter(xy[idx, 0], xy[idx, 1], s=s, c=[palette.get(x, "#7f7f7f") for x in v[idx]],
                           linewidths=0, rasterized=rasterized)
        if legend:
            from matplotlib.lines import Line2D
            present = [c for c in pd.unique(v)]
            order = sorted([c for c in present if c not in GREY_LABELS]) + sorted([c for c in present if c in GREY_LABELS])
            handles = [Line2D([], [], marker="o", ls="", color=palette.get(c, "#7f7f7f"), markersize=5, label=c) for c in order]
            ax.legend(handles=handles, fontsize=7, frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5),
                      ncol=1 if len(order) <= 14 else 2, handletextpad=0.2, columnspacing=0.6)
    else:
        vv = v.to_numpy(dtype=float)
        nan = ~np.isfinite(vv)
        ax.scatter(xy[nan, 0], xy[nan, 1], s=s, color=GREY, linewidths=0, rasterized=rasterized)
        idx = np.flatnonzero(~nan)
        idx = idx[rng.permutation(len(idx))]
        sc_ = ax.scatter(xy[idx, 0], xy[idx, 1], s=s, c=vv[idx], cmap=cmap, vmin=vmin, vmax=vmax,
                         linewidths=0, rasterized=rasterized)
        ax.figure.colorbar(sc_, ax=ax, fraction=0.045, pad=0.02)
    return ax


def before_after_grid(xy_pre, xy_post, obs_pre, obs_post, columns, *, titles=None,
                      continuous_ranges=None, panel_size=(4.4, 3.9), row_labels=("Before", "After")):
    """Rows = before / after refinement, columns = the same annotations with shared palettes and colour ranges."""
    import matplotlib.pyplot as plt
    titles = titles or columns
    continuous_ranges = continuous_ranges or {}
    fig, axes = plt.subplots(2, len(columns), figsize=(panel_size[0] * len(columns), panel_size[1] * 2),
                             squeeze=False, constrained_layout=True)
    for j, col in enumerate(columns):
        a = pd.Series(obs_pre[col]).reset_index(drop=True)
        b = pd.Series(obs_post[col]).reset_index(drop=True)
        cat = not pd.api.types.is_numeric_dtype(a)
        pal = _palette(set(a.astype(str)) | set(b.astype(str))) if cat else None
        vmin, vmax = continuous_ranges.get(col, (None, None))
        if not cat and vmin is None:
            both = np.r_[a.to_numpy(float), b.to_numpy(float)]
            both = both[np.isfinite(both)]
            if len(both):
                vmin, vmax = np.percentile(both, [1, 99])
        plot_embedding(axes[0, j], xy_pre, a, title=f"{row_labels[0]}: {titles[j]}", categorical=cat, palette=pal,
                       vmin=vmin, vmax=vmax, legend=True)
        plot_embedding(axes[1, j], xy_post, b, title=f"{row_labels[1]}: {titles[j]}", categorical=cat, palette=pal,
                       vmin=vmin, vmax=vmax, legend=False)
    return fig


def highlight_blocks(xy, blocks, *, order=None, panel_size=(3.6, 3.4), suptitle=None):
    """Small multiples: each block in colour over all other cells in grey."""
    import matplotlib.pyplot as plt
    blocks = pd.Series(blocks).astype(str).reset_index(drop=True)
    order = order or list(pd.unique(blocks))
    pal = _palette(order)
    xy = np.asarray(xy)
    s = _point_size(len(xy))
    fig, axes = plt.subplots(1, len(order), figsize=(panel_size[0] * len(order), panel_size[1]),
                             squeeze=False, constrained_layout=True)
    for ax, blk in zip(axes[0], order):
        m = (blocks == blk).to_numpy()
        ax.scatter(xy[~m, 0], xy[~m, 1], s=s, color=GREY, linewidths=0, rasterized=True)
        ax.scatter(xy[m, 0], xy[m, 1], s=s, color=pal[blk], linewidths=0, rasterized=True)
        ax.set_title(f"{blk}\n(n={m.sum():,})", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    if suptitle:
        fig.suptitle(suptitle)
    return fig


def sparsify_matrices(adata):
    """Convert a dense .X or dense layers to CSR in place (the hosted van Galen counts layer is dense int32).

    Subsetting a dense 37,627 x 21,782 layer inside a preprocessor copies ~3 GB; the CSR copy is far smaller.
    Returns the names of the matrices that were converted.
    """
    import gc
    import scipy.sparse as sp
    changed = []
    if adata.X is not None and not sp.issparse(adata.X):
        adata.X = sp.csr_matrix(adata.X)
        changed.append("X")
    for key in list(adata.layers.keys()):
        if not sp.issparse(adata.layers[key]):
            adata.layers[key] = sp.csr_matrix(adata.layers[key])
            changed.append(f"layers[{key!r}]")
    gc.collect()
    return changed


def leiden(adata, **kwargs):
    """scanpy Leiden with the igraph backend (fast, no FutureWarning); falls back to leidenalg if igraph is absent."""
    try:
        return sc.tl.leiden(adata, flavor="igraph", n_iterations=2, directed=False, **kwargs)
    except ImportError:
        return sc.tl.leiden(adata, **kwargs)


## Data, bridge, and genotype labels

A reused bridge restores its fitted preprocessors without refitting. If the bridge is built here,
preprocessors are fitted only on CITE training cells and saved with the base model. Query genotype labels
are then built exactly as in Fig. 7. Only import reference bundles you generated or trust, since fitted
preprocessors are serialized Python objects.

In [ ]:
def preprocess_dab_processed_X(
    adata,
    features,
    *,
    clip=10.0,
):
    """
    Preprocess DAb-seq when .X already contains source-processed
    continuous protein measurements.

    This intentionally does NOT:
      - look for raw counts
      - apply CLR
      - create layers['counts']

    It reproduces the Figure 7 DAb-specific path:
      processed X -> feature-wise unit standardization -> clipping
    """
    features = list(features)
    missing = [f for f in features if f not in adata.var_names]
    if missing:
        raise ValueError(
            f"DAb is missing {len(missing)} CITE reference features: "
            f"{missing[:10]}"
        )
    out = adata[:, features].copy()
    X = (
        out.X.toarray()
        if sp.issparse(out.X)
        else np.asarray(out.X)
    ).astype(np.float32)
    # DAb-specific self-standardization, as in Figure 7
    mu = np.nanmean(X, axis=0).astype(np.float32)
    sd = np.nanstd(X, axis=0).astype(np.float32)
    # Guard against invariant markers
    sd = np.where(sd < 1e-8, 1.0, sd)
    X = (X - mu) / sd
    X = np.clip(X, -clip, clip).astype(np.float32)
    out.X = X
    out.uns["adt_transform"] = {
        "type": "already_processed",
        "standardize": "unit",
        "standardize_fit": "dab_self",
        "clip": float(clip),
    }
    return out


def vg_labels(obs, gene):
    gene_u = str(gene).upper()

    mut = (
        obs["MutTranscripts"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.contains(gene_u, regex=False)
    )
    wt = (
        obs["WtTranscripts"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.contains(gene_u, regex=False)
    )

    y = np.full(len(obs), np.nan, dtype=np.float32)
    y[(mut & ~wt).to_numpy()] = 1.0
    y[(wt & ~mut).to_numpy()] = 0.0
    return y


def mutation_columns_for_gene(obs, gene):
    """Find DAb variant columns containing the gene as a token."""
    gene = str(gene).upper()
    pattern = re.compile(
        rf"(?<![A-Z0-9]){re.escape(gene)}(?![A-Z0-9])",
        flags=re.IGNORECASE,
    )

    hits = [
        col
        for col in obs.columns
        if pattern.search(str(col).upper())
    ]

    # Prefer the manuscript variants when several annotations exist.
    preference = {
        "NPM1": ["W288"],
        "DNMT3A": ["R882"],
        "FLT3": ["ITD"],
    }.get(gene, [])

    preferred = [
        col
        for col in hits
        if any(
            token in str(col).upper()
            for token in preference
        )
    ]

    return preferred if preferred else hits


def coerce_variant_call(series):
    """
    DAb mutation columns are treated as:
      missing -> NA
      zero/false/WT -> 0
      nonzero/true/mutant -> 1
    """
    s = pd.Series(series, index=series.index)

    if pd.api.types.is_bool_dtype(s):
        return s.astype(float).to_numpy()

    numeric = pd.to_numeric(s, errors="coerce")

    out = np.full(len(s), np.nan, dtype=np.float32)
    numeric_ok = numeric.notna().to_numpy()

    if numeric_ok.any():
        values = numeric.to_numpy(dtype=float)
        out[numeric_ok] = (
            values[numeric_ok] != 0
        ).astype(np.float32)

    # Fill still-unparsed string labels.
    unresolved = ~numeric_ok & s.notna().to_numpy()
    if unresolved.any():
        text = (
            s.astype("string")
            .str.strip()
            .str.lower()
        )

        wt_tokens = {
            "wt",
            "wildtype",
            "wild-type",
            "false",
            "negative",
            "neg",
            "no",
        }
        mut_tokens = {
            "mut",
            "mutant",
            "true",
            "positive",
            "pos",
            "yes",
        }

        for i in np.flatnonzero(unresolved):
            value = text.iloc[i]
            if value in wt_tokens:
                out[i] = 0.0
            elif value in mut_tokens:
                out[i] = 1.0

    return out


def dab_gene_label(obs, gene):
    cols = mutation_columns_for_gene(obs, gene)

    if not cols:
        return np.full(len(obs), np.nan, dtype=np.float32), []

    calls = np.column_stack(
        [
            coerce_variant_call(obs[col])
            for col in cols
        ]
    )

    labeled = np.isfinite(calls).any(axis=1)
    positive = np.nan_to_num(
        calls,
        nan=0.0,
    ).max(axis=1) > 0

    y = np.full(len(obs), np.nan, dtype=np.float32)
    y[labeled] = positive[labeled].astype(np.float32)

    return y, cols

In [ ]:
BRIDGE_DIR = resolve_aml_reference(
    BRIDGE_DIR, archive_path=BRIDGE_ARCHIVE_PATH,
    train_if_missing=TRAIN_BRIDGE_IF_MISSING, builder=build_base_aml_bridge,
)
bridge, prep, bridge_meta = load_reference(BRIDGE_DIR, device=device)
if not {"rna", "adt"}.issubset(prep):
    raise ValueError("The reference is missing its fitted RNA/ADT preprocessors.")
if bridge_meta.get("dataset") != "aml_mosaic":
    raise ValueError("Expected an aml_mosaic reference.")
stage = bridge_meta.get("stage")
if stage is not None and stage != "base_before_mutation_refinement":
    raise ValueError("Use the unrefined base bridge, not a mutation-refined model.")
rna_prep, adt_prep = prep["rna"], prep["adt"]
data = uds.load("aml_mosaic", keys=["vangalen_rna", "dabseq_adt"])
vg, dab = data["vangalen_rna"], data["dabseq_adt"]
del data
# The hosted van Galen counts are stored densely; converting to CSR first avoids a ~3 GB copy inside transform().
print("converted to sparse:", sparsify_matrices(vg))
vg_pp = rna_prep.transform(vg)
gc.collect()
dab_pp = preprocess_dab_processed_X(dab, features=adt_prep.features_, clip=10.0)

labels = {"vg": {g: vg_labels(vg_pp.obs, g) for g in GENES}, "dab": {}}
for g in DAB_GENES:
    labels["dab"][g], cols = dab_gene_label(dab_pp.obs, g)
    print(f"DAb {g}: {cols}")

VG_SAMPLES = vg_pp.obs["orig.ident"].astype(str).to_numpy()
VG_PATIENT = (np.array([str(VG_SAMPLE_TO_PATIENT[s]) for s in VG_SAMPLES]) if VG_SAMPLE_TO_PATIENT else VG_SAMPLES)
DAB_EXP = dab_pp.obs["experiment"].astype(str).to_numpy()
z_vg = encode_adata(bridge, vg_pp, modality="rna", device=device, latent="modality_mean")
z_dab = encode_adata(bridge, dab_pp, modality="adt", device=device, latent="modality_mean")
print("van Galen", vg_pp.shape, "| DAb", dab_pp.shape, "| latent dim", z_vg.shape[1])

## 1. What genotype information does the frozen bridge carry?

The bridge never saw a genotype label, so reading genotype out of its latent space is leakage-free. Three
questions, per cohort and gene, all with **whole samples / experiments held out**:

- **Linear probe.** How well does a logistic regression on the frozen latent rank mutant above wild-type cells?
- **Cell state alone.** How well does knowing only the cell state (van Galen `CellType`; DAb `leiden`) do, with the
  mutant rate for each state learned on other groups? Genotype and cell state are linked in AML: wild-type cells in
  a patient sample are often residual normal cells, so this baseline can be high without any genotype-specific
  signal.
- **Within strata.** AUROC computed *inside* each sample × cell-state stratum and averaged, which asks whether the
  latent separates mutant from wild-type cells that are otherwise comparable.

If the Fig. 7 notebook's cross-validation has been run, its out-of-fold mutation-head predictions are added as one
row per cross-validated arm (for example with and without anchoring). Intervals resample whole groups and are shown only when at least five groups are labeled.

The *within* column is left empty for the cell-state baseline: inside a single sample × cell-state stratum that
score is constant, so its within-stratum AUROC is 0.5 by construction.

In [ ]:
STATE = {"vg": vg_pp.obs["CellType"].astype(str).to_numpy(), "dab": dab_pp.obs["leiden"].astype(str).to_numpy()}
GROUPS = {"vg": VG_PATIENT, "dab": DAB_EXP}
Z = {"vg": z_vg, "dab": z_dab}

oof = None
if OOF_PATH.is_file():
    oof = pd.read_csv(OOF_PATH)
    print("Loaded Fig. 7 out-of-fold predictions:", OOF_PATH)
else:
    print("No Fig. 7 out-of-fold predictions found; showing the frozen-bridge analyses only.")


OOF_ARM_LABELS = {"refined": "Refined", "frozen_bn": "Refined, frozen BatchNorm", "anchored": "Refined + anchoring",
                  "shared_refined": "Shared genes, refined", "shared_anchored": "Shared genes + anchoring"}
OOF_ARMS = list(dict.fromkeys(oof["arm"])) if oof is not None and "arm" in oof else ([None] if oof is not None else [])


def oof_for(cohort, gene, arm=None):
    if oof is None or f"p_{gene}" not in oof:
        return None
    sub = oof[oof["cohort"] == cohort]
    if arm is not None:
        sub = sub[sub["arm"] == arm]
    sub = sub.set_index("cell")[f"p_{gene}"]
    names = (vg_pp if cohort == "vg" else dab_pp).obs_names.astype(str)
    return sub.reindex(names).to_numpy(dtype=float)


def auc_or_nan(y, p):
    ok = np.isfinite(y) & np.isfinite(p)
    return roc_auc_score(y[ok], p[ok]) if len(np.unique(y[ok])) == 2 else np.nan


rows = []
for cohort, genes in (("dab", DAB_GENES), ("vg", GENES)):
    grp, state = GROUPS[cohort], STATE[cohort]
    strata = np.array([f"{a}|{b}" for a, b in zip(grp, state)])
    for g in genes:
        y = labels[cohort][g]
        if np.nansum(y == 1) < 20 or np.nansum(y == 0) < 20 or len(np.unique(grp[np.isfinite(y)])) < 2:
            continue
        scores = {"linear probe (frozen bridge)": grouped_linear_probe(Z[cohort], y, grp, n_splits=5),
                  "cell state only": covariate_only_baseline(y, state, grp)}
        for arm in OOF_ARMS:
            head = oof_for(cohort, g, arm)
            if head is not None:
                name = "mutation head (Fig. 7 CV)" if arm is None else f"mutation head, {OOF_ARM_LABELS.get(arm, arm)} (Fig. 7 CV)"
                scores[name] = head
        n_groups = len(np.unique(grp[np.isfinite(y)]))
        for name, p in scores.items():
            within, _ = stratified_auroc(y, p, strata)
            row = {"cohort": cohort, "gene": g, "score": name, "n labeled": int(np.isfinite(y).sum()),
                   "groups": n_groups, "AUROC": auc_or_nan(y, p),
                   "within sample × state": np.nan if name == "cell state only" else within["weighted_auroc"],
                   "strata used": within["n_strata"]}
            if n_groups >= 5:
                ci = group_bootstrap(y, p, grp, n_boot=500)
                row["95% CI"] = f"{ci['lo']:.2f}–{ci['hi']:.2f}"
            rows.append(row)
genotype_info = pd.DataFrame(rows)
print("Pooled AUROC (whole samples held out):")
display(genotype_info.pivot_table(index=["cohort", "gene"], columns="score", values="AUROC", sort=False).round(3))
print("AUROC within sample × cell-state strata:")
display(genotype_info.dropna(subset=["within sample × state"])
        .pivot_table(index=["cohort", "gene"], columns="score", values="within sample × state", sort=False).round(3))
print("All rows, with group-bootstrap intervals where >= 5 groups are labeled:")
display(genotype_info.round(3))

**Neighbourhood structure.** kNN genotype agreement in the frozen latent, compared with a global label
shuffle and with a shuffle *within* each sample / experiment. Shuffling within groups keeps each group's mutant
fraction, so only the excess over that null reflects genotype structure beyond sample identity.

In [ ]:
enrichment = pd.DataFrame([
    {"cohort": cohort, **neighborhood_genotype_enrichment(Z[cohort], labels[cohort][HERO], GROUPS[cohort], k=30, n_perm=200)}
    for cohort in ("dab", "vg")
])
display(enrichment.round(4))

## 2. Joint clone genotypes in DAb-seq

DAb-seq calls NPM1, DNMT3A and FLT3 in the *same* cell, so each cell with all three calls has a joint genotype.
Clonal order is a useful prior here: DNMT3A mutations can arise in pre-leukemic haematopoietic stem cells that
still produce non-leukemic progeny (Shlush et al., *Nature* 2014), whereas NPM1 mutations are typically acquired
later. A testable consequence is that DNMT3A-only cells should occupy a broader or different set of latent
neighbourhoods than NPM1-mutant cells from the same experiment. The comparison is made within experiment.

Only experiments in which all three genes were genotyped are used; in the others FLT3 (or another gene) was not called, so a joint genotype is undefined. Entropy is computed over the latent clusters occupied by each genotype within an experiment (higher = spread over more neighbourhoods), for genotypes with at least 30 cells.

In [ ]:
gt = pd.DataFrame({g: labels["dab"][g] for g in ["NPM1", "DNMT3A", "FLT3"]})
complete = gt.notna().all(axis=1).to_numpy()
combo = np.full(len(gt), "incomplete", dtype=object)
combo[complete] = gt[complete].astype(int).apply(
    lambda r: "+".join([g for g in gt.columns if r[g] == 1]) or "WT (all three)", axis=1).to_numpy()
dab_pp.obs["clone_genotype"] = combo
complete_exps = sorted(set(DAB_EXP[complete]))
print("Experiments with all three genotype calls:", complete_exps)
display(pd.crosstab(DAB_EXP[complete], combo[complete], rownames=["experiment"], colnames=["joint genotype"]))

# latent neighbourhoods on the DAb block of the bridge latent
a = sc.AnnData(z_dab, obs=dab_pp.obs[["clone_genotype", "experiment", "leiden"]].copy())
sc.pp.neighbors(a, n_neighbors=30, use_rep="X"); leiden(a, resolution=0.5, key_added="z_cluster"); sc.tl.umap(a, random_state=0)
sc.pl.umap(a, color=["clone_genotype", "experiment", "z_cluster"], wspace=0.4, legend_fontsize=7)

def cluster_entropy(labels_):
    p = pd.Series(np.asarray(labels_, dtype=str)).value_counts(normalize=True).to_numpy()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

rows = []
for e in complete_exps:
    for c in ["DNMT3A", "DNMT3A+NPM1", "NPM1", "WT (all three)"]:
        m = (DAB_EXP == e) & (combo == c)
        if m.sum() >= 30:
            rows.append({"experiment": e, "genotype": c, "n": int(m.sum()),
                         "latent-cluster entropy": cluster_entropy(a.obs["z_cluster"][m])})
display(pd.DataFrame(rows).pivot(index="experiment", columns="genotype", values="latent-cluster entropy").round(2))

Protein profiles by joint genotype, within experiment (DAb features are the 18 shared surface markers):

In [ ]:
prof = pd.DataFrame(np.asarray(dab_pp.X), columns=dab_pp.var_names)
prof["genotype"], prof["experiment"] = combo, DAB_EXP
keep = prof["genotype"].isin(["DNMT3A", "DNMT3A+NPM1", "NPM1", "WT (all three)"])
# centre within experiment so the heatmap shows genotype differences, not experiment differences
num = prof.columns.difference(["genotype", "experiment"])
centred = prof.loc[keep, num] - prof.loc[keep].groupby("experiment")[list(num)].transform("mean")
centred["genotype"] = prof.loc[keep, "genotype"]
hm = centred.groupby("genotype").mean()
plt.figure(figsize=(9, 2.2)); plt.imshow(hm, cmap="bwr", vmin=-1, vmax=1, aspect="auto")
plt.yticks(range(len(hm)), hm.index); plt.xticks(range(hm.shape[1]), hm.columns, rotation=60, ha="right", fontsize=7)
plt.colorbar(label="z, centred within experiment"); plt.tight_layout(); plt.show()

## 3. Copy-number subclones from van Galen RNA

**What to expect biologically.** Copy-number inference from expression works best where large CNAs exist.
NPM1-mutated AML usually has a normal karyotype, while TP53-mutated AML is frequently associated with a complex
karyotype, so the TP53-labelled patients are the most promising place to find CNV-defined subclones in this cohort.
Expression-based calls are noisy and cannot resolve small or copy-neutral events; allele-aware tools (e.g. Numbat)
need read-level data that the hosted objects do not include.

**Gene positions.** `infercnvpy` needs `chromosome`, `start`, `end` in `.var`. The next cell downloads the GENCODE v44 basic
annotation once into the UniVI cache and reports how many genes received a position.

**Reference cells.** Use cells called normal by the authors. The hosted object has `PredictionRefined` and
`CellType`; print their values and set `NORMAL_MASK` accordingly (van Galen annotated malignant states with a
"-like" suffix, e.g. `HSC-like`, which the fallback uses).

**Reference cells.** The reference is the normal counterpart of each malignant state (HSC, Prog, GMP, ProMono, Mono, cDC), not all normal cells: comparing myeloid cells with T, B, plasma or erythroid cells makes lineage-specific gene clusters look like copy-number changes and inflates the normal-cell score distribution.

In [ ]:
import gc
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import infercnvpy as cnv

print(vg_pp.obs["PredictionRefined"].value_counts().to_string())
print(vg_pp.obs["CellType"].value_counts().to_string())

# Limit worker count and chunk size to reduce peak memory use on Windows.
CNV_N_JOBS = 1
CNV_CHUNKSIZE = 2000

# GENCODE v44 basic annotation (GRCh38), downloaded once into the UniVI cache.
GTF_URL = (
    "https://ftp.ebi.ac.uk/pub/databases/gencode/"
    "Gencode_human/release_44/gencode.v44.basic.annotation.gtf.gz"
)
GTF_PATH = uds.get_data_dir() / "gencode" / Path(GTF_URL).name

if not GTF_PATH.exists():
    GTF_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = GTF_PATH.with_suffix(".part")
    print("Downloading", GTF_URL)
    urllib.request.urlretrieve(GTF_URL, tmp)
    tmp.rename(GTF_PATH)  # Only a complete download gets the final name.

print("GTF:", GTF_PATH.name)

# Reload raw data from the local cache when rerunning this cell.
if "vg" not in globals():
    vg = uds.load("aml_mosaic", keys=["vangalen_rna"])["vangalen_rna"]
    sparsify_matrices(vg)

if not vg.obs_names.equals(vg_pp.obs_names):
    raise ValueError(
        "Raw and preprocessed van Galen cells are in a different order."
    )

# Lean object: raw counts plus the metadata used below.
counts_key = getattr(rna_prep, "layer", None)
counts = (
    vg.layers[counts_key]
    if counts_key and counts_key in vg.layers
    else vg.X
)

cnv_ad = sc.AnnData(
    X=sp.csr_matrix(counts, dtype=np.float32),
    obs=vg.obs[["PredictionRefined", "CellType", "orig.ident"]].copy(),
    var=pd.DataFrame(index=vg.var_names.copy()),
)

del counts, vg
gc.collect()

# Annotate genomic positions and retain mapped genes.
cnv.io.genomic_position_from_gtf(
    GTF_PATH,
    cnv_ad,
    gtf_gene_id="gene_name",
)

mapped = cnv_ad.var["chromosome"].notna()
print(f"genes with a genomic position: {mapped.mean():.1%}")
cnv_ad = cnv_ad[:, mapped].copy()

# Each run starts from raw counts, so normalization happens once.
sc.pp.normalize_total(cnv_ad, target_sum=1e4)
sc.pp.log1p(cnv_ad)

# Reference cells: normal cells of the lineages the malignant cells resemble. van Galen names malignant states
# "<state>-like" (HSC-like, GMP-like, ...) next to their normal counterparts (HSC, GMP, ...). Lymphoid, plasma and
# erythroid cells are left out of the reference: their lineage-specific gene clusters (immunoglobulin, HLA,
# haemoglobin loci, ...) can look like copy-number changes when myeloid cells are compared with them.
pred = cnv_ad.obs["PredictionRefined"].astype(str).str.lower()
ctype = cnv_ad.obs["CellType"].astype(str)
malignant_states = sorted({t for t in ctype.unique() if t.endswith("-like")})
REFERENCE_STATES = sorted({t[: -len("-like")] for t in malignant_states} & set(ctype.unique()))
NORMAL_MASK = ((pred == "normal") & ctype.isin(REFERENCE_STATES)).to_numpy()
if NORMAL_MASK.sum() < 200:
    print("Too few lineage-matched normal cells; falling back to all normal cells as the reference.")
    NORMAL_MASK = (pred == "normal").to_numpy()
    REFERENCE_STATES = sorted(set(ctype[NORMAL_MASK]))
if not NORMAL_MASK.any():
    raise ValueError("No normal reference cells were identified.")
print("reference states:", REFERENCE_STATES, f"({int(NORMAL_MASK.sum()):,} cells)")

cnv_ad.obs["cnv_ref"] = pd.Categorical(
    np.where(NORMAL_MASK, "normal", "query")
)
print("reference cells:", int(NORMAL_MASK.sum()))

# Fix Arrow-backed gene-name indexing inside infercnvpy.
# Explicit object dtype keeps the underlying index values as a NumPy array.
cnv_ad.var_names = pd.Index(
    cnv_ad.var_names.to_numpy(dtype=object),
    dtype=object,
    name=cnv_ad.var_names.name,
)


gc.collect()

cnv.tl.infercnv(
    cnv_ad,
    reference_key="cnv_ref",
    reference_cat=["normal"],
    window_size=100,
    step=10,
    n_jobs=CNV_N_JOBS,
    chunksize=CNV_CHUNKSIZE,
)

cnv.tl.pca(cnv_ad)
cnv.pp.neighbors(cnv_ad)
try:
    cnv.tl.leiden(cnv_ad, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
except ImportError:
    cnv.tl.leiden(cnv_ad, resolution=0.5)
cnv.tl.cnv_score(cnv_ad)


The CNV profile of every cell relative to the normal reference (collapsed above with the progress logs):

In [ ]:
sub_idx = np.sort(np.random.default_rng(0).choice(cnv_ad.n_obs, min(8000, cnv_ad.n_obs), replace=False))
cnv.pl.chromosome_heatmap(cnv_ad[sub_idx].copy(), groupby="cnv_ref")

**CNV burden per sample.** A cell's CNV score is compared with the 99th percentile of the (lineage-matched) reference cells'
scores, so about 1% of normal cells exceed it by construction. For each sample, the fraction of *malignant* cells
above that threshold measures CNV burden. Samples where at least `CNV_HIGH_FRACTION` of malignant cells exceed it are
treated as CNV-high, and only there are malignant cells clustered on their CNV profiles into candidate subclones; a
subclone is kept if it has at least 30 cells and its median score exceeds the threshold.

Most AML, including most NPM1-mutated AML, has a normal karyotype, so few samples are expected to be CNV-high;
TP53-mutated AML is the main exception, and samples with TP53-mutant calls are marked below.

In [ ]:
CNV_HIGH_FRACTION = 0.25

cnv_ad.obs["patient"] = VG_PATIENT
score = cnv_ad.obs["cnv_score"].to_numpy()
malignant = (cnv_ad.obs["PredictionRefined"].astype(str).str.lower() == "malignant").to_numpy()
CNV_THRESHOLD = float(np.quantile(score[NORMAL_MASK], 0.99))
print(f"99th percentile of reference-cell CNV scores: {CNV_THRESHOLD:.4f}")

rows = []
for p in np.unique(VG_PATIENT):
    mp = VG_PATIENT == p
    mal, nor = mp & malignant, mp & NORMAL_MASK
    if mal.sum() < 50:
        continue
    rows.append({"sample": p, "malignant cells": int(mal.sum()), "reference cells": int(nor.sum()),
                 "fraction above threshold": float((score[mal] > CNV_THRESHOLD).mean()),
                 "median score, malignant": float(np.median(score[mal])),
                 "TP53-mutant calls": int(np.nansum(labels["vg"]["TP53"][mp] == 1))})
burden = pd.DataFrame(rows).sort_values("fraction above threshold", ascending=False).reset_index(drop=True)
display(burden.round(3))

fig, ax = plt.subplots(figsize=(max(6, 0.28 * len(burden)), 3.2), constrained_layout=True)
ax.bar(range(len(burden)), burden["fraction above threshold"],
       color=np.where(burden["TP53-mutant calls"] > 0, "#c44e52", "#4c72b0"))
ax.axhline(CNV_HIGH_FRACTION, color="k", ls=":", lw=1, label="CNV-high cut-off")
ax.axhline(0.01, color="grey", ls="--", lw=1, label="expected for normal cells")
ax.set_xticks(range(len(burden))); ax.set_xticklabels(burden["sample"], rotation=90, fontsize=6)
ax.set_xlabel("van Galen sample (red: TP53-mutant calls)"); ax.set_ylabel("malignant cells above threshold")
ax.legend(fontsize=7)
plt.show()

CNV_HIGH = [str(x) for x in burden.loc[burden["fraction above threshold"] >= CNV_HIGH_FRACTION, "sample"]]
print("CNV-high samples:", CNV_HIGH or "none")

cnv_ad.obs["cnv_subclone"] = "NA"
cnv_ad.obs["subclone_ok"] = False
X_cnv_pca = np.asarray(cnv_ad.obsm["X_cnv_pca"])
sub_rows = []
for p in CNV_HIGH:
    m = (VG_PATIENT == p) & malignant
    sub = sc.AnnData(X_cnv_pca[m])
    sc.pp.neighbors(sub, n_neighbors=15, use_rep="X")
    leiden(sub, resolution=0.3)
    labels_p = np.array([f"{p}:c{c}" for c in sub.obs["leiden"]])
    cnv_ad.obs.loc[m, "cnv_subclone"] = labels_p
    for lab in np.unique(labels_p):
        in_lab = np.flatnonzero(m)[labels_p == lab]
        med = float(np.median(score[in_lab]))
        keep = len(in_lab) >= 30 and med > CNV_THRESHOLD
        cnv_ad.obs.iloc[in_lab, cnv_ad.obs.columns.get_loc("subclone_ok")] = keep
        sub_rows.append({"subclone": lab, "cells": len(in_lab), "median score": med, "kept": keep})
if sub_rows:
    display(pd.DataFrame(sub_rows).round(4))
    show = np.isin(VG_PATIENT, CNV_HIGH[:4]) & malignant
    ref_idx = np.random.default_rng(0).choice(np.flatnonzero(NORMAL_MASK), min(2000, int(NORMAL_MASK.sum())), replace=False)
    sel = show.copy(); sel[ref_idx] = True
    view = cnv_ad[sel].copy()
    view.obs["group"] = pd.Categorical(np.where(NORMAL_MASK[sel], "normal reference", "sample " + view.obs["patient"].astype(str)))
    cnv.pl.chromosome_heatmap(view, groupby="group")
else:
    print("No sample reaches the CNV-high cut-off, so no CNV subclones are defined; this is consistent with the "
          "largely normal-karyotype AML expected in this cohort.")

**Overlap with mutation calls.** For each patient with at least two confident subclones, test whether mutant
transcript detection differs between subclones (Fisher's exact test). Detection depends on expression of the
mutated gene, so compare the fraction of *covered* cells (mutant or wild-type call) as well; a subclone with more
coverage is not evidence of more mutation.

In [ ]:
tests = []
for p in np.unique(VG_PATIENT):
    m = (cnv_ad.obs["patient"] == p).to_numpy() & cnv_ad.obs["subclone_ok"].to_numpy()
    clones = cnv_ad.obs.loc[m, "cnv_subclone"].value_counts()
    clones = clones[clones >= 30].index
    for g in labels["vg"]:
        y = labels["vg"][g]
        for i, c1 in enumerate(clones):
            for c2 in clones[i + 1:]:
                a1 = (cnv_ad.obs["cnv_subclone"] == c1).to_numpy(); a2 = (cnv_ad.obs["cnv_subclone"] == c2).to_numpy()
                t = [[np.nansum(y[a1] == 1), np.nansum(y[a1] == 0)], [np.nansum(y[a2] == 1), np.nansum(y[a2] == 0)]]
                if min(sum(t[0]), sum(t[1])) >= 5:
                    tests.append({"patient": p, "gene": g, "clone_a": c1, "clone_b": c2,
                                  "mut/cov a": f"{t[0][0]}/{sum(t[0])}", "mut/cov b": f"{t[1][0]}/{sum(t[1])}",
                                  "coverage a": np.mean(np.isfinite(y[a1])), "coverage b": np.mean(np.isfinite(y[a2])),
                                  "fisher_p": fisher_exact(t)[1]})
tests = pd.DataFrame(tests)
if len(tests):
    from statsmodels.stats.multitest import multipletests
    tests["fdr"] = multipletests(tests["fisher_p"], method="fdr_bh")[1]
    display(tests.sort_values("fdr").head(20))
else:
    print("No patient has two CNV subclones with enough mutation-call coverage to test.")

**Where CNV subclones sit in the latent space.** If the bridge latent separates CNV subclones of the same
patient, expression differences between subclones are large enough for the RNA encoder to resolve; if it does not,
subclone identity is not recoverable from this latent without CNV-aware features.

In [ ]:
from sklearn.metrics import silhouette_score
rows = []
for p in np.unique(VG_PATIENT):
    m = (cnv_ad.obs["patient"] == p).to_numpy() & cnv_ad.obs["subclone_ok"].to_numpy()
    labs = cnv_ad.obs.loc[m, "cnv_subclone"].to_numpy()
    if m.sum() >= 60 and len(np.unique(labs)) >= 2:
        rows.append({"patient": p, "n": int(m.sum()), "n_subclones": len(np.unique(labs)),
                     "silhouette in UniVI latent": silhouette_score(z_vg[m], labs),
                     "silhouette in CNV space": silhouette_score(np.asarray(cnv_ad.obsm["X_cnv_pca"])[m], labs)})
if rows:
    display(pd.DataFrame(rows).round(3))
else:
    print("No sample has two or more kept CNV subclones, so subclone separation cannot be assessed.")

u = sc.AnnData(z_vg, obs=cnv_ad.obs[["cnv_score", "CellType"]].copy())
u.obs[f"{HERO} call"] = np.select([labels["vg"][HERO] == 1, labels["vg"][HERO] == 0], ["MUT", "WT"], "Unlabeled")
sc.pp.neighbors(u, use_rep="X", n_neighbors=30); sc.tl.umap(u, random_state=0)
u.obs["CNV-high sample"] = np.where(np.isin(VG_PATIENT, CNV_HIGH), "CNV-high", "other")
sc.pl.umap(u, color=["cnv_score", "CellType", f"{HERO} call", "CNV-high sample"], wspace=0.4, legend_fontsize=6, ncols=2,
           vmax="p99")

## 4. Sampling genotype-defined regions of the latent space

`fit_label_latent_gaussians` fits one **diagonal** Gaussian per label. A genotype class that spans several cell
states is multimodal in the latent, so a single Gaussian puts mass between the states, where there are no cells.
Two remedies are compared here: condition the label on cell state (`CellType|genotype`), and sample from the
*aggregated posterior* instead, i.e. pick real cells of the group and draw z ~ q(z|x)
(`encode_adata(..., latent="modality_sample")`). Full posterior draws add the encoder's whole uncertainty and tend to land off the manifold, so draws with the noise
scaled down (temperature τ = 0.5 and 0.25: z = μ + τ·σ·ε) are shown too. `knn_distance_ratio` near 1 means the
samples lie on the manifold.

**Held-out evaluation.** The mutant cells are split in half. Gaussians are fitted and posterior draws taken from half A only; distances and the two-sample test use half B. Comparing generated cells with the cells they were drawn from would reward copying (a C2ST AUROC below 0.5 is the tell-tale sign).

In [ ]:
y = labels["vg"][HERO]
state = vg_pp.obs["CellType"].astype(str).to_numpy()
geno = np.select([y == 1, y == 0], ["MUT", "WT"], "NA")
lab_sg = np.array([f"{s}|{g}" for s, g in zip(state, geno)])

# Half A of the mutant cells is used to fit / sample; half B only to evaluate.
mut_all = np.random.default_rng(0).permutation(np.flatnonzero(geno == "MUT"))
MUT_A, MUT_B = np.sort(mut_all[: len(mut_all) // 2]), np.sort(mut_all[len(mut_all) // 2:])
fit_mask = geno != "NA"
fit_mask[MUT_B] = False
print(f"mutant cells: {len(MUT_A)} for fitting / sampling, {len(MUT_B)} held out for evaluation")

g1 = fit_label_latent_gaussians(z_vg[fit_mask], geno[fit_mask], min_n=20)
g2 = fit_label_latent_gaussians(z_vg[fit_mask], lab_sg[fit_mask], min_n=20)
N = 1000
samples = {}
if "MUT" in g1:
    samples["Gaussian on genotype"] = sample_latent_by_label(g1, "MUT", N, random_state=0)
# state-conditioned Gaussian mixture, weighted by the number of half-A MUT cells in each state
sg = {k: v for k, v in g2.items() if k.endswith("|MUT")}
if sg:
    w = np.array([sg[k]["n"] for k in sg], float); w /= w.sum()
    counts = np.random.default_rng(0).multinomial(N, w)
    samples["Gaussian on state|genotype"] = np.vstack([sample_latent_by_label(g2, k, int(c), random_state=i)
                                                       for i, (k, c) in enumerate(zip(sg, counts)) if c > 0])
else:
    print("No cell state has >= 20 half-A mutant cells, so the state-conditioned sampler is skipped.")
# posterior draws for half-A mutant cells, with the encoder noise scaled by a temperature
mut_cells = vg_pp[MUT_A].copy()
reps = int(np.ceil(N / mut_cells.n_obs))
mu_mut = encode_adata(bridge, mut_cells, modality="rna", device=device, latent="modality_mean")
draws = [encode_adata(bridge, mut_cells, modality="rna", device=device, latent="modality_sample", random_state=r)
         for r in range(reps)]
pick = np.random.default_rng(1).choice(reps * mut_cells.n_obs, N, replace=False)
for tau in (1.0, 0.5, 0.25):
    tempered = np.vstack([mu_mut + tau * (d - mu_mut) for d in draws])
    samples[f"posterior draws, τ = {tau}"] = tempered[pick]

z_eval = z_vg[MUT_B]
display(pd.DataFrame({k: knn_distance_ratio(v, z_eval) for k, v in samples.items()}).T.round(3))

**Decode and compare with decoded real cells.** Decoders here are Gaussian on train-scaled log1p RNA, so
decoded values are z-scores in the CITE reference scaling; `rna_prep.scaler_.inverse_transform` maps them back to
log1p. The two-sample test compares decoded samples with *decoded real cells* (same decoder), which isolates the
sampling step from decoder smoothing. Remember that the decoders were trained only on the Knorr et al. CITE-seq
bridge: generated profiles live on that cohort's manifold, not on van Galen's.

In [ ]:
def decode_rna_log1p(z):
    x = generate_from_latent(bridge, z=np.asarray(z, np.float32), target_mod="rna", device=device)
    return rna_prep.scaler_.inverse_transform(x) if getattr(rna_prep, "scaler_", None) is not None else x

x_eval_dec = decode_rna_log1p(z_eval)
genes = np.asarray(rna_prep.features_)
rng_c2st = np.random.default_rng(2)
report = []
for k, zs in samples.items():
    x = decode_rna_log1p(zs[rng_c2st.choice(len(zs), len(z_eval), replace=False)])   # balanced with half B
    report.append({"sampler": k, "C2ST AUROC vs held-out MUT cells (0.5 = indistinguishable)": c2st_auc(x_eval_dec, x),
                   "weighted LSC17 (mean)": lsc17_weighted(x, genes, standardize=False)["score"].mean()})
report.append({"sampler": "held-out MUT cells (decoded)", "C2ST AUROC vs held-out MUT cells (0.5 = indistinguishable)": np.nan,
               "weighted LSC17 (mean)": lsc17_weighted(x_eval_dec, genes, standardize=False)["score"].mean()})
display(pd.DataFrame(report).round(3))

## 5. A cross-modal counterfactual: genotype direction from DAb protein, read out as RNA

DAb-seq has dense, per-cell genotype labels but only 18 proteins; van Galen has transcriptomes but sparse labels.
Because both are embedded in one latent space, a genotype direction estimated from DAb can be applied to van Galen
cells and decoded to RNA. The direction is estimated **within experiment × cluster strata**, so it is not simply
the difference between patients or between blasts and residual normal cells; `per_stratum` shows how consistent
it is across strata (cosine to the pooled direction).

**Positive controls for NPM1.** NPM1-mutant AML characteristically has high HOXA/HOXB and MEIS1 expression
(Brunetti et al., *Cancer Cell* 2018) and is often CD34-negative (Falini et al., *NEJM* 2005). If the counterfactual
shift raises HOXA9/MEIS1 and lowers CD34, the direction carries known NPM1 biology; if not, it is probably tracking
something else. Either way the result is a hypothesis to check against real cells, done in the last cell.

The per-stratum table reports how well each stratum's direction agrees with the pooled one (cosine); widely varying cosines mean the genotype effect is not consistent across experiments and cell states, and the counterfactual should be read with that in mind.

In this cohort many of the largest counterfactual increases are ribosomal-protein genes, so the direction is also shown with the latent's ribosomal-content axis projected out; a genotype effect that survives this is less likely to be a proxy for ribosomal content or cell size.

In [ ]:
strata = np.array([f"{e}|{c}" for e, c in zip(DAB_EXP, dab_pp.obs["leiden"].astype(str))])
dirn = stratified_mean_difference_direction(z_dab, labels["dab"][HERO], strata, min_per_class=20)
display(dirn["per_stratum"].sort_values("cosine_to_pooled").round(3))
d = dirn["direction"]

# Ribosomal-content axis of the van Galen latent: the latent shift per unit of the fraction of counts from
# ribosomal-protein genes. The genotype direction is also evaluated with that axis projected out.
counts_vg = vg_pp.layers["counts"]
rp_genes = vg_pp.var_names.str.match(r"^(RPL|RPS)\d")
rb_frac = (np.asarray(counts_vg[:, rp_genes].sum(axis=1)).ravel()
           / np.maximum(np.asarray(counts_vg.sum(axis=1)).ravel(), 1.0))
coef = np.linalg.lstsq(np.c_[np.ones(len(rb_frac)), rb_frac], z_vg, rcond=None)[0][1]
if rp_genes.sum() == 0 or not np.isfinite(coef).all() or np.linalg.norm(coef) == 0:
    print("No ribosomal-protein genes / no ribosomal axis found; the orthogonalised direction equals the original.")
    r_hat, d_orth = np.zeros_like(d), d.copy()
else:
    r_hat = coef / np.linalg.norm(coef)
    d_orth = d - (d @ r_hat) * r_hat
    d_orth = d_orth / np.linalg.norm(d_orth)
    print(f"cosine(genotype direction, ribosomal-content axis) = {float(d @ r_hat):.3f}")

# apply to van Galen wild-type-called malignant cells; step = 1 SD of the projection onto the direction
malignant_state = pd.Series(state).str.endswith("-like").to_numpy()   # van Galen malignant-state naming
base = np.flatnonzero((geno == "WT") & malignant_state)
base = base if len(base) >= 50 else np.flatnonzero(geno == "WT")
x0 = decode_rna_log1p(z_vg[base])


def counterfactual_effect(direction, alphas=(0.5, 1.0, 2.0)):
    sd = float(np.std(z_vg @ direction))
    return pd.DataFrame({a: pd.Series((decode_rna_log1p(z_vg[base] + a * sd * direction) - x0).mean(0), index=genes)
                         for a in alphas})


EXPECTED = {"HOXA9": "up", "HOXA10": "up", "HOXB3": "up", "MEIS1": "up", "CD34": "down"}


def control_table(e):
    pct = e[1.0].rank(pct=True) * 100            # 100 = largest increase
    t = pd.DataFrame({"Δ log1p (α=1)": e[1.0], "percentile": pct}).reindex([g for g in EXPECTED if g in e.index])
    t["matches expected"] = [(v > 0) == (EXPECTED[g] == "up") for g, v in t["Δ log1p (α=1)"].items()]
    return t


eff = counterfactual_effect(d)
eff_orth = counterfactual_effect(d_orth)
print("Positive controls for NPM1 (expected: HOXA / HOXB / MEIS1 up, CD34 down); percentile 100 = most increased")
display(pd.concat({"genotype direction": control_table(eff), "ribosome axis removed": control_table(eff_orth)},
                  axis=1).round(3))

housekeeping = eff.index.str.match(r"^(RPL|RPS|MT-|MRPL|MRPS)")
for name, e in (("genotype direction", eff), ("ribosome axis removed", eff_orth)):
    top50 = e[1.0].rank(ascending=False) <= 50
    print(f"\n{name}: ribosomal / mitochondrial genes among the 50 largest increases: {int((housekeeping & top50).sum())}")
    print("  top 20 up (excl. RP/MT):  ", e.loc[~housekeeping, 1.0].nlargest(20).index.tolist())
    print("  top 20 down (excl. RP/MT):", e.loc[~housekeeping, 1.0].nsmallest(20).index.tolist())

**Check the counterfactual against real cells.** Within van Galen patient × cell-state strata that contain
both NPM1-called and wild-type-called cells, compute the real mean log1p difference per gene and correlate it with
the counterfactual effect. A positive Spearman correlation on held-out real data is the minimum bar before any gene
from the generated profiles is interpreted.

In [ ]:
logx = vg_pp.layers["log1p"]
diffs, weights = [], []
pstrata = np.array([f"{p}|{c}" for p, c in zip(VG_PATIENT, state)])
for s in np.unique(pstrata):
    m = pstrata == s
    m1, m0 = m & (y == 1), m & (y == 0)
    if m1.sum() >= 5 and m0.sum() >= 5:
        diffs.append(np.asarray(logx[m1].mean(0)).ravel() - np.asarray(logx[m0].mean(0)).ravel())
        weights.append(1 / (1 / m1.sum() + 1 / m0.sum()))
if diffs:
    real = pd.Series(np.average(np.vstack(diffs), axis=0, weights=weights), index=vg_pp.var_names).reindex(eff.index)
    ok = real.notna()
    expressed = pd.Series(np.asarray(logx.mean(axis=0)).ravel() > 0.1, index=vg_pp.var_names).reindex(eff.index).fillna(False)
    rows = []
    for dname, e in (("genotype direction", eff), ("ribosome axis removed", eff_orth)):
        for label, sel in (("all genes", ok), ("expressed genes (mean log1p > 0.1)", ok & expressed)):
            rho, p = spearmanr(e.loc[sel, 1.0], real[sel])
            rows.append({"direction": dname, "genes": label, "n": int(sel.sum()), "Spearman rho": rho, "p": p})
    display(pd.DataFrame(rows).round(4))
    print(f"strata used: {len(diffs)}")
else:
    print("No van Galen stratum has >=5 NPM1-called and >=5 wild-type-called cells.")

## 6. What these analyses support

* **Genotype information in the frozen bridge (section 1):** the comparison between the linear probe, cell state
  alone, and the within-stratum AUROC shows how much of any genotype prediction is carried by cell state and sample
  composition. Only the within-stratum number speaks to genotype differences among comparable cells.
* **Joint genotypes (DAb):** differences in latent spread or protein profile between DNMT3A-only and NPM1-mutant
  cells *within an experiment* are observations about this cohort; they are consistent or inconsistent with the
  clonal-order prior, not proof of it.
* **CNV subclones:** interpretable only for subclones whose CNV score clears the normal-cell threshold; association
  with mutant-transcript calls must be read alongside coverage.
* **Generated cells:** the realism metrics say whether samples sit on the learned manifold. Generated profiles are
  interpolations of the Knorr et al. bridge decoder, so they cannot reveal states absent from that cohort.
* **Counterfactual direction:** associative, not causal. It is worth reporting only if the positive controls move
  in the expected direction and the effect correlates with real within-stratum differences.